# ESS.112 and telescope elevation study


## References

- [Times Square: Image Quality Nightly Report: IQ, AOS, and EAS](https://usdf-rsp.slac.stanford.edu/times-square/github/lsst-sitcom/ts_aos_analysis/notebooks/nightly_report/nightly_report_aos_eas?dayobs=20250827&ts_hide_code=1)
- [GitHub: Image Quality Nightly Report: IQ, AOS, and EAS](https://github.com/lsst-sitcom/ts_aos_analysis/blob/develop/notebooks/nightly_report/nightly_report_aos_eas.ipynb)
- [SITCOM-2212 Investigate possible correlation between temperature from ESS:112 and telescope elevation](https://rubinobs.atlassian.net/browse/SITCOM-2212)
- [SITCOM-2079 Traceback and model for the available temperature sensors during ComCam](https://rubinobs.atlassian.net/browse/SITCOM-2079)
- [LTS-192 M2 Electronics Design Document](https://docushare.lsstcorp.org/docushare/dsweb/Get/Document-54702/LTS-192%20M2%20Electronics%20Design%20Document.pdf)


## Introduction

I expect this notebook to be a little different from other's I have been working on.  
Instead of being a notebook where you simply execute all the cells,  
I expect it to be more like a study.  
Let's see how it goes.

The main challenge with this study is that the temperatures are different every day.  
So we cannot make a simple and direct correlation between the temperature measured on ESS:112 and the telescope elevation.  
Let me start then my analysis be querying the telemetry displayed in [Nightly Report EAS in Times Square].  
Then, I will consider the data from a full day obs for now.  
I will create a correlation matrix using binned data with the min and max values groupy by each half second.
  
[Nightly Report EAS in Times Square]: https://usdf-rsp.slac.stanford.edu/times-square/github/lsst-sitcom/ts_aos_analysis/notebooks/nightly_report/nightly_report_aos_eas?dayobs=20250827&ts_hide_code=1


## Single Night Analysis

As mentioned above, let's start with a single day obs and let's start with the telemetry in [Nightly Report EAS in Times Square].

In [ ]:
z = ["alpha","bravo","charlie"]
new_z = [i[0]*2 for i in z]
print(new_z)

In [ ]:
# Let's use this for single-day analysis
day_obs = 20250827à
log_level = "INFO"

In [ ]:
import asyncio
import matplotlib as mpl
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytz

from astropy.coordinates import get_sun, AltAz, EarthLocation
from astropy.time import Time, TimeDelta
from typing import List

from lsst_efd_client import EfdClient
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)

import logging

In [ ]:
# Create an instance of the EfdClient
efd_client = makeEfdClient()

# Gather the start and end time of our analysis
start_time = getDayObsStartTime(day_obs)
end_time = getDayObsEndTime(day_obs)

# Create a logger for debugging later
logger = logging.getLogger("ess112_elevation_study")
logger.setLevel(log_level)

print(f"This initial analysis will contain data from:\n"
      f"  {start_time} to {end_time}.\n")

### ESS:112 M2; RPi with sticker 1

The [ESS:112](https://github.com/lsst-ts/ts_config_ocs/blob/49e7dd64b62415685f28b8bc2f179a3633c05059/ESS/v8/_init.yaml#L227) is one of our favorite sensors that we use to estimate the temperature inside the dome.  
This is the sensor in which we believe that is measuring temperature gradients depending on the elevation angle. 

Previous analysis showed that the average sampling is about 1.4 seconds.  
For now, I will keep the original sampling.  
Once I learn more about the other datasets, I can select a different sampling rate.

In [ ]:
ess_112_df = await efd_client.select_time_series(
    topic_name="lsst.sal.ESS.temperature", 
    fields=["temperatureItem0"], 
    start=start_time, 
    end=end_time, 
    index=112
)

In [ ]:
ess_112_df.plot(title="Inside Temperature - ESS:112", figsize=(10, 3))
plt.grid(":", alpha=0.2)

We can see lots of spikes in this data.  
This is probably what Elana is looking for. 

Part of the analysis is to look for correlation between these temperatures and the elevation angle.  
We can plot both together to see if the peaks have correlation with any elevation change.  
Let's do this.

In [ ]:
query = efd_client.build_time_range_query(
    "lsst.sal.MTMount.elevation",
    ["mean(actualPosition)"],
    start_time,
    end_time,
) + "GROUP BY time(30s)"

elevation_df = await efd_client.influx_client.query(query)

Since we now have a function defined above that allows us to easily find the end and beginng of the astronomical twilight,  
let's plot them together with everything else to see if that reagion is a good candidate for the thermalized region.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

fig.suptitle("ESS:112 Temperature and Telescope Elevation")
ax.plot(ess_112_df, color="C0", label="ESS:112 Temperature")
ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(loc="upper right")

ax2 = ax.twinx()
ax2.plot(elevation_df, color="C1", label="Telescope Elevation")
ax2.set_ylabel("Elevation Angle [deg]")
ax2.legend(loc="lower left")

plt.show()

Yes, it seems quite fair. I can add these same lines to the other plots below.

### ESS:301 Weather Tower

[ESS:301] can give us some reference about the environment conditions.  
We know, by experience, that the temperature inside and outsite the dome can be very different.  
I might drop this one later if we decide to focus more in our analysis.  
Remember that the goal here is to try to identify a vertical gradient inside the dome.

The sampling average here is about 3.8 seconds.  
You can easily see that we will have to do some data-processing if we want to do our correlation matrix. 

[ESS:301]: https://github.com/lsst-ts/ts_config_ocs/blob/49e7dd64b62415685f28b8bc2f179a3633c05059/ESS/v8/_init.yaml#L480

In [ ]:
ess_301_df = await efd_client.select_time_series(
    topic_name="lsst.sal.ESS.temperature", 
    fields=["temperatureItem0"], 
    start=start_time, 
    end=end_time, 
    index=301
)

In [ ]:
ess_301_df.plot(title="Outside Temperature - Weather Tower", figsize=(10, 3))
plt.grid(":", alpha=0.2)

Here you can clearly see the effect of the sun in the temperature outside.  
There is some fluctuation, but the overal trend is very clear.  
This is typical of a dataset that is more robust and insensitive to other parameters, as it should be.

### MTM2.temperature

The MTM2 CSC have internal temperature sensors as well.  
The [MTM2.temperature] contains 12 channels corresponding to 12 sensors inside the cell.  
Based on a quick conversation with Elana, it seems that column `ring5` corresponds to sensor 
that is mounted temporarily outside the m2 cell.  
This is why we can also use it to track ambient temperature. 

The sampling for this dataset has an average of 0.05 seconds.  
This is much more than what we need.  
A query containing 24h of telemetry can take almost five minutes to be completed.  
Considering the time scale of the telemetry above, 
and considering that we are still in an exploratory phase, let's do a sampling of half minute.

[MTM2.temperature]: https://ts-xml.lsst.io/sal_interfaces/MTM2.html#temperature

In [ ]:
# First, let's build the query
query = efd_client.build_time_range_query(
    "lsst.sal.MTM2.temperature",
    [f"mean(ring{i}) as mean_ring_{i}" for i in range(12)],
    start_time,
    end_time,
) + "GROUP BY time(30s)"

# And then we query it, this takes a long time
mtm2_temp_df = await efd_client.influx_client.query(query)

<br>
Just because I am curious, let's have a look at the data considering all the channels.

In [ ]:
m2rings_colors = ["darkblue", "royalblue", "cornflowerblue", "lightskyblue",
          "darkslategray", "darkcyan", "turquoise", "mediumseagreen", 
          "indigo", "darkviolet", "violet", "deeppink"]
mtm2_temp_df.plot(title="M2 Temperatures", figsize=(10, 5), color=m2rings_colors)
plt.legend(ncols=3, loc="lower right")
plt.grid(":", alpha=0.2)

It calls my attention that `ring5` is the topic that we are using.  
It seems sensitive to "something" else. 

All the other temperature channels have a clear trend.  
Some of them show some oscillation (like ring 6 and ring 4).

### Thermalized time range

The plots above show how the temperature changes during the day pollute lots of what we are looking for.  
As a first rule of thumb, I will select the timestamps between the end and the beginning of astronomical twilights.  
I hope this can be used to minimize the effects of temperature changes due to the day/night cycle in my analysis.  
  
I extracted the code for the `find_sun_altitude_crossings` function from the [Times Square: Image Quality Nightly Report: IQ, AOS, and EAS](https://usdf-rsp.slac.stanford.edu/times-square/github/lsst-sitcom/ts_aos_analysis/notebooks/nightly_report/nightly_report_aos_eas?dayobs=20250827&ts_hide_code=1) notebook.  
It will be useful later when we start batch data analysis with several night.  

In [ ]:
def find_sun_altitude_crossings(start_time: Time, end_time: Time, altitude_deg: float, rising: bool) -> list:
    """
    Return a list of localized datetimes when the sun crosses a given altitude
    (e.g., -18 for twilight, 0 for sunrise/sunset) between start_time and end_time.

    Parameters
    ----------
    start_time : Time
        Astropy start time.
    end_time : Time
        Astropy end time.
    altitude_deg : float
        Target sun altitude in degrees.
    rising : bool
        True for rising (upward crossing), False for setting (downward).

    Returns
    -------
    list[datetime]
        Localized CLT datetimes when the sun crosses the given altitude.
    """
    location = EarthLocation.of_site("Cerro Pachon")
    clt = pytz.timezone("America/Santiago")

    step_seconds = 60
    n_steps = int((end_time - start_time).sec / step_seconds) + 1
    times = start_time + TimeDelta(np.arange(n_steps) * step_seconds, format='sec')

    altaz = AltAz(obstime=times, location=location)
    sun_alt = get_sun(times).transform_to(altaz).alt.deg
    times_clt = times.to_datetime(timezone=clt)

    crossings = []
    for i in range(1, len(sun_alt)):
        prev_alt = sun_alt[i - 1]
        curr_alt = sun_alt[i]
        if rising and prev_alt < altitude_deg <= curr_alt:
            crossings.append(times_clt[i])
        elif not rising and prev_alt > altitude_deg >= curr_alt:
            crossings.append(times_clt[i])

    return crossings

<br>
For now, let's repeat the ESS:112 and ESS:301 plots together with the end and the begining of the astronomical twilights. <br> 
This will give us an impression about using this time window as a reference.  

In [ ]:
twilight_ends = find_sun_altitude_crossings(start_time, end_time, altitude_deg=-18, rising=False)[0]
twilight_starts = find_sun_altitude_crossings(start_time, end_time, altitude_deg=-18, rising=True)[0]

print(f"The end of the astronomical twilight on day_obs = {day_obs} was at {twilight_ends}.")
print(f"The begin of the astronomical twilight on day_obs = {day_obs} was at {twilight_starts}.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

fig.suptitle("ESS:112/ESS:301 Temperature and Telescope Elevation")

ax.axvline(twilight_ends, c="k", ls=":", alpha=0.5, label="End of the astronomical twilight")
ax.axvline(twilight_starts, c="k", ls="--", alpha=0.5, label="Begin of the astronomical twilight")
ax.plot(ess_112_df, color="C0", label="ESS:112 Inside Temperature")
ax.plot(ess_301_df, color="C2", label="ESS:301 Outside Temperature")

ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(loc="upper right", fontsize="small")

ax2 = ax.twinx()
ax2.plot(elevation_df, color="C1", label="Telescope Elevation")
ax2.set_ylabel("Elevation Angle [deg]")
ax2.legend(loc="lower left", fontsize="small")

fig.autofmt_xdate()
plt.show()

<br>
Let's make a zoom on the area we area interested in.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

fig.suptitle("ESS:112/ESS:301 Temperature and Telescope Elevation")

ax.axvline(twilight_ends, c="k", ls=":", alpha=0.5, label="End of the astronomical twilight")
ax.axvline(twilight_starts, c="k", ls="--", alpha=0.5, label="Begin of the astronomical twilight")
ax.plot(ess_112_df, color="C0", label="ESS:112 Inside Temperature")
ax.plot(ess_301_df, color="C2", label="ESS:301 Outside Temperature")

ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(loc="upper right", fontsize="small")
ax.set_xlim(twilight_ends, twilight_starts)

ax2 = ax.twinx()
ax2.plot(elevation_df, color="C1", label="Telescope Elevation")
ax2.set_ylabel("Elevation Angle [deg]")
ax2.legend(loc="lower left", fontsize="small")

fig.autofmt_xdate()
plt.show()

Yup, it seems fair.  
Let's now work on querying the data with a consistent sampling and concatenate all the data frames.

### Consolidated Table

In [ ]:
async def single_sampled_query(
    client: EfdClient, 
    topic: str,
    columns: List[str],
    begin: Time,
    end: Time, 
    sal_index: int | None=None,
    sampling: str="10s"
    ):

    query = client.build_time_range_query(topic, columns, begin, end, index=sal_index)
    query += f" GROUP BY time({sampling})"
    logger.debug(f"Query for {topic}:\n {query}")

    df = await client.influx_client.query(query)
    df.dropna(inplace=True)

    return df
    

async def query_and_concat_data(_efd_client: EfdClient, _day_obs: int, sampling: str="10s"):

    # Let's start with the time stamps
    start_day_obs_time = getDayObsStartTime(_day_obs)
    end_day_obs_time = getDayObsEndTime(_day_obs)

    start_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=False)[0]
    end_night_time = find_sun_altitude_crossings(
        start_day_obs_time, end_day_obs_time, altitude_deg=-18, rising=True)[0]

    start_night_time = Time(start_night_time, scale="utc")
    end_night_time = Time(end_night_time, scale="utc")

    dfs = []

    # We will keep queries separated to it is easier to refactor later
    
    queries = [
        # ESS:112 Inside Temperature 
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS112"],    
            sal_index=112
        ),
        # ESS:301 Outside Temperature
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS301"],
            sal_index=301
        ),
        # M2 Temperatures 
        dict(
            topic="lsst.sal.MTM2.temperature",
            columns=[f"mean(ring{i}) as mean_ring_{i}" for i in range(12)],
            sal_index=None
        ),
        # Telescope elevation 
        dict(
            topic="lsst.sal.MTMount.elevation",
            columns=["mean(actualPosition) as tel_elevation"],
            sal_index=None
        )
    ]

    # Run the queries
    for q in queries:
        temp_df = await single_sampled_query(
            _efd_client,
            q["topic"],
            q["columns"],
            start_night_time,
            end_night_time,
            sal_index=q["sal_index"]
        )
        dfs.append(temp_df)
            
    # Concatenate everything together
    output_df = pd.concat(dfs, axis=1)
    return output_df


df = await query_and_concat_data(efd_client, day_obs)
df

### Correlation Heatmap

Alright! Now we are in a good shape to create our correlation matrix!  
Let's have a look at it.

In [ ]:
# From Matplotlib - Annotaded Heatmaps
# https://matplotlib.org/stable/gallery/images_contours_and_fields/image_annotated_heatmap.html#using-the-helper-function-code-style
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="", **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (M, N).
    row_labels
        A list or array of length M with the labels for the rows.
    col_labels
        A list or array of length N with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current Axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
    cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=-30, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    ax.tick_params(top=True, bottom=False,
                   labeltop=True, labelbottom=False)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = mpl.ticker.StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

In [ ]:
fig, ax1 = plt.subplots(num="Correlation Map", figsize=(8,8))

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 9)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

im, cbar = heatmap(
    df.corr(), 
    df.columns, 
    df.columns, 
    ax=ax1, 
    cbarlabel="Correlation", 
    cmap=cmap,
    norm=norm,
    cbar_kw=dict(
        ticks=np.linspace(-1, 1, 9), 
    )
)

annotate_heatmap(
    im=im,
    data=df.corr(),
    fontsize="x-small"
)

fig.suptitle(f"Correlation Matrix - Day Obs {day_obs}")
plt.show()

This looks great.  
But it might be too soon for any conclusion.  
There are several weird things going on here.  
For example, what is wrong with `mean_ring_1`?  
I will think later if I want to keep all of the M2 ring temperature telemetry or if I will drop it.  

For now, let me expand this analysis for multiple nights.

## Multiple Nights Analysis

### Select nights fully on sky

I have been going back and forth in this analysis.  
As I look at the plots, I believe it is hard to evaluate what affects what if we include the Sun in the equation.  
I want to consider the scenario where the dome is partially thermalized to make analysis easier.  

Let's start picking up our dates.  

In [ ]:
day_obs_start = 20250501
day_obs_end = 20250831

<br>  

For this work, I will use the [ConsDB_visits_metadata] notebook in Times Square.  
The code seems to have exactly what I need. 

The cells bellow require the [lsst-sims/rubin_nights] module, which seems not to be in the standard stack.  
The first of them will install it using PIP. 
You can either comment it out or simply skip it if you are sure you have it running.

[ConsDB_visits_metadata]: https://usdf-rsp.slac.stanford.edu/times-square/github/lsst/schedview_notebooks/nightly/ConsDB_visits_metadata?day_obs_min=20250620&day_obs_max=20250820&instrument=lsstcam&ts_hide_code=1
[lsst-sims/rubin_nights]: https://github.com/lsst-sims/rubin_nights

In [ ]:
import os

if os.getenv("EXTERNAL_INSTANCE_URL") is not None:
    print("updating rubin_nights\n")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git  --no-deps  > /dev/null 2>&1

In [ ]:
import logging
from rubin_nights import connections
from rubin_nights.observatory_status import get_dome_open_close


# Try to silence the annoying logs 
logging.getLogger("httpx").setLevel(logging.CRITICAL)
logging.getLogger("rubin_nights.connections").setLevel(logging.CRITICAL)

<br> 

All the functions and classes in [lsst-sims/rubin_nights] use a sync version of the EFD client.  
Since I am lazy and I don't want to rewrite the code, I will create an instance of this version and use it.

[lsst-sims/rubin_nights]: https://github.com/lsst-sims/rubin_nights

In [ ]:
sync_efd_client = connections.get_clients()['efd']

<br>

We can now finally get the dome status.  
In this case, our data frame will have a single row.  
But, in future analysis, we will have several rows. 

In [ ]:
dome_status = get_dome_open_close(
    getDayObsStartTime(day_obs_start), 
    getDayObsEndTime(day_obs_end), 
    sync_efd_client
)

dome_status.head(5)

Cool! Now we can select the `day_obs` that have more than ten hours per night.  
This is a good start. We can think of a better criteria later.

In [ ]:
logger.info(f"Number of nights between {day_obs_start} and {day_obs_end}: {long_nights.index.size}")
long_nights = dome_status[dome_status.open_hours >= 8]

logger.info(f"Number of nights in that interval with open hours >= 8: {long_nights.index.size}")
long_nights.head(5)

### Input table

Now it is the moment to build a table that we will use to query all the relevant data.  
We want a dataframe in which each row corresponds to a day obs.  
Alright, now let's complement our dataframe with some new columns that will be useful!

<br>
Arg! For whatever reason, I need to have the code in the cells below split and ron cell-by-cell. 

In [ ]:
# There is some annoying error related to indexes and slicing 
# that go away with this cell. I don't understand why and I don't want to 
# spend time on it now.
long_nights = long_nights.copy()

In [ ]:
long_nights.loc[:, "start_time"] = long_nights["day_obs"].apply(
    getDayObsStartTime)

In [ ]:
long_nights.loc[:, "end_time"] = long_nights["day_obs"].apply(
    getDayObsEndTime)

In [ ]:
long_nights.loc[:, "end_of_astronomical_twilight"] = long_nights.apply(
    lambda row: find_sun_altitude_crossings(
        row['start_time'], row['end_time'], altitude_deg=-18, rising=False)[0], axis=1)

In [ ]:
long_nights.loc[:, "begin_of_astronomical_twilight"] = long_nights.apply(
    lambda row: find_sun_altitude_crossings(
        row['start_time'], row['end_time'], altitude_deg=-18, rising=True)[0], axis=1)

In [ ]:
long_nights.head(5)

### Batch query

Now we can query all the data.  
This will be lots of data and it might take a while.  
So, be patient. 

In [ ]:
async def query_row_data(_efd_client: EfdClient, row: pd.Series) -> pd.DataFrame:
    """
    Given a row associated with an day_obs, query all the relevant telemetry
    """
    start_night_time = Time(row["end_of_astronomical_twilight"])
    end_night_time = Time(row["begin_of_astronomical_twilight"])
    dfs = []

    # We will keep queries separated to it is easier to refactor later
    queries = [
        # ESS:112 Inside Temperature 
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS112"],    
            sal_index=112
        ),
        # ESS:301 Outside Temperature
        dict(
            topic="lsst.sal.ESS.temperature",
            columns=[f"mean(temperatureItem0) as temperature_ESS301"],
            sal_index=301
        ),
        # M2 Temperatures 
        dict(
            topic="lsst.sal.MTM2.temperature",
            columns=[f"mean(ring{i}) as mean_ring_{i}" for i in range(12)],
            sal_index=None
        ),
        # Telescope elevation 
        dict(
            topic="lsst.sal.MTMount.elevation",
            columns=["mean(actualPosition) as tel_elevation"],
            sal_index=None
        )
    ]

    # Run the queries
    for q in queries:
        temp_df = await single_sampled_query(
            _efd_client,
            q["topic"],
            q["columns"],
            start_night_time,
            end_night_time,
            sal_index=q["sal_index"]
        )
        dfs.append(temp_df)
            
    # Concatenate everything together
    output_df = pd.concat(dfs, axis=1)
    return output_df


async def query_batch_data(_efd_client: EfdClient, df: pd.DataFrame) -> pd.DataFrame:
    """
    Iterate over all the nights, collect telemetry, 
    and concatenate all the results.
    """
    logger.debug(f"Number of rows in the input dataframe: {df.index.size}")
    dfs = []
    
    for index, row in df.iterrows():
        logger.info(f"Processing row: {index}")
        _temp_df = await query_row_data(_efd_client, row)
        dfs.append(_temp_df)
        logger.info(f"Completed row: {index}")

    logger.debug(f"Number of dataframes to be concatenated: {len(dfs)}")
    output_df = pd.concat(dfs, axis=0)
    
    return output_df


results_df = await query_batch_data(efd_client, long_nights)
results_df

### Batch Heat Map


In [ ]:
fig, ax1 = plt.subplots(num="Batch Correlation Map", figsize=(8,8))

cmap = mpl.cm.Spectral
bounds = np.linspace(-1, 1, 9)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N, extend='both')

im, cbar = heatmap(
    results_df.corr(), 
    results_df.columns, 
    results_df.columns, 
    ax=ax1, 
    cbarlabel="Correlation", 
    cmap=cmap,
    norm=norm,
    cbar_kw=dict(
        ticks=np.linspace(-1, 1, 9), 
    )
)

annotate_heatmap(
    im=im,
    data=df.corr(),
    fontsize="x-small"
)

fig.suptitle(f"Batch Correlation Matrix\n Day Obs {day_obs_start} - {day_obs_end}")
plt.show()